In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

# Create a unified Gold streaming table
dp.create_streaming_table(
    name="all_taxi_trips",
    comment="Unified Yellow and Green taxi trips from Silver",
    table_properties={"delta.feature.timestampNtz": "supported"}
)

def normalize_trips(df, taxi_type, pickup_column, dropoff_column):
    return (
        df.select(
            "year", "month", "day",
            F.lit(taxi_type).alias("taxi_type"),
            F.col(pickup_column).alias("pickup_datetime"),
            F.col(dropoff_column).alias("dropoff_datetime"),
            "PULocationID", "DOLocationID", "passenger_count", "trip_distance",
            "payment_type", "fare_amount", "tip_amount", "tolls_amount", "total_amount"
        )
        .withColumn("trip_duration_minutes", F.expr("timestampdiff(SECOND, pickup_datetime, dropoff_datetime) / 60.0"))
    )

# Append Yellow Taxi records
@dp.append_flow(target="all_taxi_trips", name="yellow_taxi_flow")
def yellow_taxi_flow():
    return normalize_trips(
        spark.readStream.table("silver.yellow_trips"),
        taxi_type="yellow",
        pickup_column="tpep_pickup_datetime",
        dropoff_column="tpep_dropoff_datetime"
    )

# Append Green Taxi records
@dp.append_flow(target="all_taxi_trips", name="green_taxi_flow")
def green_taxi_flow():
    return normalize_trips(
        spark.readStream.table("silver.green_trips"),
        taxi_type="green",
        pickup_column="lpep_pickup_datetime",
        dropoff_column="lpep_dropoff_datetime"
    )




In [0]:
# Create dashboard-ready route metrics
@dp.materialized_view(
    name="daily_route_metrics",
    comment="Daily taxi demand, revenue, route and payment statistics"
)
def daily_route_metrics():
    trips_df = spark.read.table("all_taxi_trips")
    taxi_zones_df = spark.read.table("bronze.taxi_zones_raw")

    pickup_zones_df = taxi_zones_df.select(
        F.col("LocationID").alias("pickup_location_id"),
        F.col("Borough").alias("pickup_borough"),
        F.col("Zone").alias("pickup_zone")
    )

    dropoff_zones_df = taxi_zones_df.select(
        F.col("LocationID").alias("dropoff_location_id"),
        F.col("Borough").alias("dropoff_borough"),
        F.col("Zone").alias("dropoff_zone")
    )

    enriched_trips_df = (
        trips_df
        .join(
            F.broadcast(pickup_zones_df),
            F.col("PULocationID") == F.col("pickup_location_id"),
            "left"
        )
        .join(
            F.broadcast(dropoff_zones_df),
            F.col("DOLocationID") == F.col("dropoff_location_id"),
            "left"
        )
        .drop("pickup_location_id", "dropoff_location_id")
    )

    route_metrics_df = (
        enriched_trips_df
        .groupBy(
            "year", "month", "day", "taxi_type",
            "pickup_borough", "pickup_zone",
            "dropoff_borough", "dropoff_zone"
        )
        .agg(
            F.count("*").alias("trip_count"),
            F.round(F.sum("total_amount"), 2).alias("total_revenue"),
            F.round(F.avg("total_amount"), 2).alias("avg_total_amount"),
            F.round(F.sum("tip_amount"), 2).alias("total_tips"),
            F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
            F.round(F.avg("trip_duration_minutes"), 2).alias("avg_trip_duration_minutes"),
            F.sum("trip_distance").alias("total_trip_distance"),
            F.sum(F.when(F.col("payment_type") == 1, F.col("fare_amount")).otherwise(0)).alias("credit_card_fare_total"),
            F.sum(F.when(F.col("payment_type") == 1, F.col("tip_amount")).otherwise(0)).alias("credit_card_tip_total"),
            F.sum(F.when(F.col("payment_type") == 1, 1).otherwise(0)).alias("credit_card_trip_count"),
            F.sum(F.when(F.col("payment_type") == 2, 1).otherwise(0)).alias("cash_trip_count")
        )
    )

    payment_count = F.col("credit_card_trip_count") + F.col("cash_trip_count")

    return (
        route_metrics_df
        .withColumn(
            "recorded_tip_rate_percentage",
            F.when(
                F.col("credit_card_fare_total") > 0,
                F.round(F.col("credit_card_tip_total") / F.col("credit_card_fare_total") * 100, 2)
            )
        )
        .withColumn(
            "net_revenue_per_mile",
            F.when(
                F.col("total_trip_distance") > 0,
                F.round(F.col("total_revenue") / F.col("total_trip_distance"), 2)
            )
        )
        .withColumn(
            "credit_card_percentage",
            F.when(
                payment_count > 0,
                F.round(F.col("credit_card_trip_count") / payment_count * 100, 2)
            )
        )
        .withColumn(
            "cash_percentage",
            F.when(
                payment_count > 0,
                F.round(F.col("cash_trip_count") / payment_count * 100, 2)
            )
        )
    )